## 1. Importación de librerías

Se utilizan **Pandas** y **NumPy** para la manipulación y el análisis de datos, **Plotly** para la visualización interactiva 3D, y utilidades estándar (`json`, `pathlib`) para la construcción del panel interactivo con sprites.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, IFrame, HTML
import json
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
import plotly
print("Plotly:", plotly.__version__)


Pandas: 2.3.3
NumPy: 2.3.5
Plotly: 6.3.0


## 2. Carga del dataset

**Origen de los datos:** el archivo `Pokemon2.csv` contiene las estadísticas base de los
Pokémon de las **generaciones 1 a 6**, compiladas originalmente a partir de
[pokemondb.net](https://pokemondb.net) (metodología equivalente al repositorio público
[`lgreski/pokemonData`](https://github.com/lgreski/pokemonData)). El archivo incluye,
además, dos registros personalizados (`Sesni`, `Chuchin`) agregados como práctica del curso.

**Contenido:** 807 registros con el número de Pokédex, nombre, tipo primario y
secundario, seis estadísticas base (HP, Ataque, Defensa, Ataque especial, Defensa
especial y Velocidad), el total de estadísticas, la generación, si es legendario y su
nivel de evolución.

**Sprites:** las imágenes de cada Pokémon se obtienen del repositorio público
[`PokeAPI/sprites`](https://github.com/PokeAPI/sprites) en GitHub (carpeta
`sprites/pokemon/`), indexadas por el número de Pokédex nacional.


In [4]:
RUTA_ORIGEN = "Pokemon2.csv"
df_raw = pd.read_csv(RUTA_ORIGEN)
df_raw.head()


,dex_number,name,type_1,type_2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary,evolution_level,generacion_valida,promedio_estadisticas,sprite_url
0,16,Pidgey,Normal,Flying,251,40,45,40,35,35,56,1,False,1,True,41.83,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,Normal,Flying,349,63,60,55,50,50,71,1,False,1,True,58.17,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,Normal,Flying,479,83,80,75,70,70,101,1,False,2,True,79.83,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,Normal,Flying,579,83,80,80,135,80,121,1,False,3,True,96.50,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,Normal,Sin segundo tipo,253,30,56,35,25,35,72,1,False,1,True,42.17,https://raw.githubusercontent.com/PokeAPI/spri...


## 3. Inspección inicial del dataset

In [5]:
df_raw.head()


,dex_number,name,type_1,type_2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary,evolution_level,generacion_valida,promedio_estadisticas,sprite_url
0,16,Pidgey,Normal,Flying,251,40,45,40,35,35,56,1,False,1,True,41.83,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,Normal,Flying,349,63,60,55,50,50,71,1,False,1,True,58.17,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,Normal,Flying,479,83,80,75,70,70,101,1,False,2,True,79.83,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,Normal,Flying,579,83,80,80,135,80,121,1,False,3,True,96.50,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,Normal,Sin segundo tipo,253,30,56,35,25,35,72,1,False,1,True,42.17,https://raw.githubusercontent.com/PokeAPI/spri...


In [6]:
print("Dimensiones (filas, columnas):", df_raw.shape)


Dimensiones (filas, columnas): (803, 17)


In [7]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 803 entries, 0 to 802
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dex_number             803 non-null    int64  
 1   name                   803 non-null    object 
 2   type_1                 803 non-null    object 
 3   type_2                 803 non-null    object 
 4   total                  803 non-null    int64  
 5   hp                     803 non-null    int64  
 6   attack                 803 non-null    int64  
 7   defense                803 non-null    int64  
 8   sp_atk                 803 non-null    int64  
 9   sp_def                 803 non-null    int64  
 10  speed                  803 non-null    int64  
 11  generation             803 non-null    int64  
 12  legendary              803 non-null    bool   
 13  evolution_level        803 non-null    int64  
 14  generacion_valida      803 non-null    bool   
 15  promed

In [8]:
df_raw.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
dex_number,803.0,NaN,NaN,NaN,363.270237,209.099012,1.0,184.5,365.0,540.5,723.0
name,803,802,Caterpie,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_1,803,19,Water,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_2,803,19,Sin segundo tipo,387,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total,803.0,NaN,NaN,NaN,435.058531,120.791407,180.0,330.0,450.0,515.0,787.0
hp,803.0,NaN,NaN,NaN,69.291407,25.569274,1.0,50.0,65.0,80.0,255.0
attack,803.0,NaN,NaN,NaN,79.84807,38.960743,5.0,55.0,75.0,100.0,676.0
defense,803.0,NaN,NaN,NaN,74.841843,38.663987,5.0,50.0,70.0,90.0,678.0
sp_atk,803.0,NaN,NaN,NaN,73.816936,40.147734,10.0,49.5,65.0,95.0,688.0
sp_def,803.0,NaN,NaN,NaN,72.890411,36.100724,20.0,50.0,70.0,90.0,678.0


## 4. Limpieza y normalización de columnas y categorías

Se normalizan los nombres de columnas (minúsculas, sin espacios ni puntos) y se
homogeneízan los valores de texto de las columnas categóricas (`Type 1`, `Type 2`),
eliminando espacios sobrantes y unificando el formato *Título*.


In [9]:
df = df_raw.copy()

# Normalización de nombres de columnas
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(".", "", regex=False)
    .str.replace(" ", "_")
)
df = df.rename(columns={"#": "dex_number"})
print("Columnas normalizadas:")
print(df.columns.tolist())


Columnas normalizadas:
['dex_number', 'name', 'type_1', 'type_2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation', 'legendary', 'evolution_level', 'generacion_valida', 'promedio_estadisticas', 'sprite_url']


In [10]:
# Normalización de valores categóricos (tipos de Pokémon)
for col in ["type_1", "type_2"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["name"] = df["name"].astype("string").str.strip()

print("Tipos únicos (type_1) tras normalizar:")
print(sorted(df["type_1"].unique()))


Tipos únicos (type_1) tras normalizar:
['Artificial', 'Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']


## 5. Tratamiento de valores nulos, duplicados y datos incorrectos

**Antes de la limpieza:**


In [11]:
print("Filas antes de limpiar:", len(df))
print("\nValores nulos por columna (antes):")
print(df.isnull().sum())
print("\nRegistros duplicados exactos (antes):", df.duplicated().sum())


Filas antes de limpiar: 803

Valores nulos por columna (antes):
dex_number               0
name                     0
type_1                   0
type_2                   0
total                    0
hp                       0
attack                   0
defense                  0
sp_atk                   0
sp_def                   0
speed                    0
generation               0
legendary                0
evolution_level          0
generacion_valida        0
promedio_estadisticas    0
sprite_url               2
dtype: int64

Registros duplicados exactos (antes): 0


**Duplicados exactos:** se encontraron 4 filas totalmente duplicadas
(`Squirtle`, `Caterpie`, `Weedle`, `Butterfree`), producto de un error de carga del
archivo original. Se eliminan conservando la primera aparición.

**Valores nulos en `type_2`:** no son un error — representan Pokémon de un **solo
tipo**. Se sustituyen por la etiqueta explícita `"Sin segundo tipo"` en lugar de
eliminarlos, para no perder información de los Pokémon monotipo.

**Datos incorrectos (generación fuera de rango):** se detectaron **2 registros**
(`Sesni` #722 y `Chuchin` #723) con generación `67` y `32` respectivamente y un tipo
`"Artificial"` que no pertenece al roster canónico de tipos Pokémon. Dado que el rango
válido de generaciones en este dataset es 1–6, se marcan como registros **especiales**
(no se eliminan, para preservar la información), pero se **excluyen del eje de
generación** en la visualización principal.


In [12]:
# --- Duplicados ---
filas_antes = len(df)
duplicados = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados eliminados: {duplicados}  |  Filas: {filas_antes} -> {len(df)}")

# --- Nulos en type_2 ---
nulos_antes = df["type_2"].isnull().sum()
df["type_2"] = df["type_2"].fillna("Sin segundo tipo")
print(f"Nulos en type_2: {nulos_antes} -> {df['type_2'].isnull().sum()}")

# --- Datos incorrectos: generaciones fuera de rango ---
generaciones_validas = list(range(1, 7))
df["generacion_valida"] = df["generation"].isin(generaciones_validas)
print("\nRegistros con generación fuera de rango (1-6):")
display(df.loc[~df["generacion_valida"], ["dex_number","name","type_1","generation","legendary"]])


Duplicados eliminados: 0  |  Filas: 803 -> 803
Nulos en type_2: 0 -> 0

Registros con generación fuera de rango (1-6):


,dex_number,name,type_1,generation,legendary
801,723,Chuchin,Artificial,32,True
802,722,Sesni,Artificial,67,True


In [13]:
print("Estado final:")
print("Filas:", len(df))
print("Nulos totales:", df.isnull().sum().sum())
print("Duplicados restantes:", df.duplicated().sum())


Estado final:
Filas: 803
Nulos totales: 2
Duplicados restantes: 0


## 6. Selección de variables estadísticas

Se seleccionan las **seis estadísticas base** oficiales de cada Pokémon —
`hp`, `attack`, `defense`, `sp_atk`, `sp_def` y `speed` — porque son las variables
numéricas fundamentales que determinan el desempeño de un Pokémon en batalla y son,
además, los componentes que originalmente conforman la columna `total`. Usarlas de
forma individual (en lugar de solo `total`) permite construir un promedio interpretable
y compararlas entre sí más adelante.


In [14]:
stats_cols = ["hp", "attack", "defense", "sp_atk", "sp_def", "speed"]
df[["name"] + stats_cols + ["total"]].head()


,name,hp,attack,defense,sp_atk,sp_def,speed,total
0,Pidgey,40,45,40,35,35,56,251
1,Pidgeotto,63,60,55,50,50,71,349
2,Pidgeot,83,80,75,70,70,101,479
3,PidgeotMega Pidgeot,83,80,80,135,80,121,579
4,Rattata,30,56,35,25,35,72,253


## 7. Creación de la columna `promedio_estadisticas`

In [15]:
df["promedio_estadisticas"] = df[stats_cols].mean(axis=1).round(2)
df[["name", "type_1", "generation"] + stats_cols + ["promedio_estadisticas"]].head()


,name,type_1,generation,hp,attack,defense,sp_atk,sp_def,speed,promedio_estadisticas
0,Pidgey,Normal,1,40,45,40,35,35,56,41.83
1,Pidgeotto,Normal,1,63,60,55,50,50,71,58.17
2,Pidgeot,Normal,1,83,80,75,70,70,101,79.83
3,PidgeotMega Pidgeot,Normal,1,83,80,80,135,80,121,96.50
4,Rattata,Normal,1,30,56,35,25,35,72,42.17


## 8. Análisis estadístico descriptivo

In [17]:
dfv = df[df["generacion_valida"]].copy()  # se excluyen los 2 registros especiales

resumen = dfv[stats_cols + ["promedio_estadisticas"]].agg(["mean", "median", "min", "max", "std"]).T
resumen.columns = ["media", "mediana", "minimo", "maximo", "desv_estandar"]
resumen.round(2)


,media,mediana,minimo,maximo,desv_estandar
hp,69.23,65.0,1.0,255.0,25.53
attack,78.94,75.0,5.0,190.0,32.48
defense,73.79,70.0,5.0,230.0,31.19
sp_atk,72.75,65.0,10.0,194.0,32.76
sp_def,71.84,70.0,20.0,230.0,27.87
speed,68.25,65.0,5.0,180.0,29.05
promedio_estadisticas,72.47,75.0,30.0,130.0,20.03


In [18]:
print("Promedio de estadísticas por generación:")
display(dfv.groupby("generation")["promedio_estadisticas"].mean().round(2))

print("\nPromedio de estadísticas por tipo principal (top 5):")
display(dfv.groupby("type_1", observed=True)["promedio_estadisticas"]
        .mean().round(2).sort_values(ascending=False).head())

print("\nLegendarios vs. no legendarios:")
display(dfv.groupby("legendary")["promedio_estadisticas"].mean().round(2))


Promedio de estadísticas por generación:


generation
1    70.90
2    69.71
3    72.70
4    76.50
5    72.50
6    72.73
Name: promedio_estadisticas, dtype: float64


Promedio de estadísticas por tipo principal (top 5):


type_1
Dragon     91.76
Steel      81.28
Flying     80.84
Psychic    79.32
Fire       76.35
Name: promedio_estadisticas, dtype: float64


Legendarios vs. no legendarios:


legendary
False     69.54
True     105.11
Name: promedio_estadisticas, dtype: float64

## 9. Preparación y orden de las variables `generación` y `tipo`

Se define un orden categórico para `type_1` (siguiendo el orden tradicional de la
Pokédex) para que la visualización agrupe los tipos de forma consistente en el eje Y,
y se ordena el DataFrame por generación, tipo y número de Pokédex.


In [19]:
orden_tipos = [
    "Normal", "Fire", "Water", "Electric", "Grass", "Ice", "Fighting",
    "Poison", "Ground", "Flying", "Psychic", "Bug", "Rock", "Ghost",
    "Dragon", "Dark", "Steel", "Fairy",
]
tipos_presentes = [t for t in orden_tipos if t in dfv["type_1"].unique()]

dfv["type_1"] = pd.Categorical(dfv["type_1"], categories=tipos_presentes, ordered=True)
dfv = dfv.sort_values(["generation", "type_1", "dex_number"]).reset_index(drop=True)

print("Orden de tipos utilizado en el eje Y:")
print(tipos_presentes)


Orden de tipos utilizado en el eje Y:
['Normal', 'Fire', 'Water', 'Electric', 'Grass', 'Ice', 'Fighting', 'Poison', 'Ground', 'Flying', 'Psychic', 'Bug', 'Rock', 'Ghost', 'Dragon', 'Dark', 'Steel', 'Fairy']


## 10. Obtención y validación de los sprites

Los sprites se construyen a partir del número de Pokédex (`dex_number`), apuntando al
repositorio [`PokeAPI/sprites`](https://github.com/PokeAPI/sprites):

```
https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{dex_number}.png
```

Los dos registros especiales (`Sesni`, `Chuchin`) no cuentan con un sprite oficial real
(usar su número de Pokédex mostraría, incorrectamente, el sprite de un Pokémon de la
generación 7), por lo que se les asigna `None` y la visualización los reemplaza por un
ícono de marcador de posición (❔).


In [20]:
SPRITE_BASE = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{}.png"

def construir_sprite(row):
    if not row["generacion_valida"]:
        return None
    return SPRITE_BASE.format(int(row["dex_number"]))

dfv["sprite_url"] = dfv.apply(construir_sprite, axis=1)
dfv[["dex_number", "name", "sprite_url"]].head()


,dex_number,name,sprite_url
0,16,Pidgey,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,https://raw.githubusercontent.com/PokeAPI/spri...


In [21]:
# Validación de accesibilidad de una muestra de sprites
import requests
muestra = dfv["sprite_url"].dropna().sample(5, random_state=42)
for url in muestra:
    try:
        r = requests.head(url, timeout=10)
        print(r.status_code, url)
    except Exception as e:
        print("ERROR", url, e)


200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/643.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/544.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/121.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/438.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/130.png


## 11. Primera versión del Scatter Plot 3D

Se construye una primera versión con `plotly.express.scatter_3d`, usando la
**generación** en el eje X, el **tipo** en el eje Y y el **promedio de estadísticas**
en el eje Z, coloreando por tipo.


In [22]:
dfv["type_1_str"] = dfv["type_1"].astype(str)

fig_v1 = px.scatter_3d(
    dfv,
    x="generation",
    y="type_1_str",
    z="promedio_estadisticas",
    color="type_1_str",
    hover_name="name",
    category_orders={"type_1_str": tipos_presentes},
    title="Versión 1 · Scatter 3D: Generación, Tipo y Promedio de estadísticas",
    labels={"generation": "Generación", "type_1_str": "Tipo", "promedio_estadisticas": "Promedio de estadísticas"},
    opacity=0.85,
    height=650,
)
fig_v1.update_traces(marker=dict(size=4))
fig_v1.show()


## 12. Diferenciación visual por tipo (color y símbolo)

Se asigna a cada tipo un **color** (paleta oficial aproximada de tipos Pokémon) y un
**símbolo** de marcador distinto (ciclando entre los símbolos disponibles en
`scatter3d`, ya que Plotly solo ofrece 8 símbolos 3D nativos frente a 18 tipos), de
forma que la combinación color+símbolo permita distinguir cada tipo incluso en
escala de grises.


In [23]:
COLOR_TIPO = {
    "Normal": "#A8A878", "Fire": "#F08030", "Water": "#6890F0",
    "Electric": "#F8D030", "Grass": "#78C850", "Ice": "#98D8D8",
    "Fighting": "#C03028", "Poison": "#A040A0", "Ground": "#E0C068",
    "Flying": "#A890F0", "Psychic": "#F85888", "Bug": "#A8B820",
    "Rock": "#B8A038", "Ghost": "#705898", "Dragon": "#7038F8",
    "Dark": "#705848", "Steel": "#B8B8D0", "Fairy": "#EE99AC",
}
SIMBOLOS = ["circle", "diamond", "square", "cross", "x", "circle-open", "diamond-open", "square-open"]
SIMBOLO_TIPO = {t: SIMBOLOS[i % len(SIMBOLOS)] for i, t in enumerate(tipos_presentes)}

dfv["color"] = dfv["type_1_str"].map(COLOR_TIPO)
dfv["simbolo"] = dfv["type_1_str"].map(SIMBOLO_TIPO)

pd.DataFrame({"tipo": tipos_presentes,
              "color": [COLOR_TIPO[t] for t in tipos_presentes],
              "simbolo": [SIMBOLO_TIPO[t] for t in tipos_presentes]})



,tipo,color,simbolo
0,Normal,#A8A878,circle
1,Fire,#F08030,diamond
2,Water,#6890F0,square
3,Electric,#F8D030,cross
4,Grass,#78C850,x
5,Ice,#98D8D8,circle-open
6,Fighting,#C03028,diamond-open
7,Poison,#A040A0,square-open
8,Ground,#E0C068,circle
9,Flying,#A890F0,diamond


## 13. Información emergente (hover) y sprites integrados

La versión final de la visualización se construye como una página HTML autónoma
(Plotly.js + JavaScript) que:

- Muestra en el **hover** de cada punto: nombre, tipo, generación y promedio de
  estadísticas.
- Incorpora un **panel lateral interactivo** que, al pasar el cursor sobre un punto,
  despliega el **sprite** del Pokémon (obtenido de `PokeAPI/sprites`), su tipo (con
  insignia de color), si es legendario y una barra por cada estadística base.
- Incluye **filtros interactivos** por generación (botones), por tipo (chips —también
  funcionan como leyenda) y por rango de promedio de estadísticas (doble control
  deslizante).
- Personaliza título, etiquetas de los ejes, tamaño, opacidad, cámara 3D y
  proporción de aspecto (`aspectratio`).

A continuación se genera y se muestra esa visualización interactiva:


In [9]:
from pathlib import Path
from PIL import Image
import pandas as pd, json, base64

BASE = Path.cwd()
CSV = BASE / "pokemon_limpio_artificial.csv"
SPRITES = BASE / "sprites"
NORMAL = BASE / "sprites_normalizados"
NORMAL.mkdir(exist_ok=True)

for p in SPRITES.glob("*.png"):
    im=Image.open(p).convert("RGBA")
    bbox=im.getchannel("A").getbbox()
    if bbox: im=im.crop(bbox)
    r,g,b,a=im.split(); a=a.point(lambda v: 0 if v < 8 else v); im=Image.merge("RGBA",(r,g,b,a))
    scale=min(78/im.width,78/im.height); nw,nh=max(1,round(im.width*scale)),max(1,round(im.height*scale))
    im=im.resize((nw,nh),Image.Resampling.LANCZOS)
    c=Image.new("RGBA",(96,96),(0,0,0,0)); c.alpha_composite(im,((96-nw)//2,min(96-nh,max(0,(96-nh)//2+2))))
    c.save(NORMAL/p.name,optimize=True)

df=pd.read_csv(CSV)
mask=df["origen"].astype(str).str.lower().eq("artificial")
df.loc[mask,"sprite_local"]=df.loc[mask,"sprite_local"].map(lambda x:f"sprites_normalizados/{Path(str(x)).name}")
df.loc[mask,"tipo_visual"]=df.loc[mask,"type_1"]
df.to_csv(CSV,index=False)
records=[]
for _,r in df.iterrows():
    records.append({
      "dex":int(r.dex_number),"name":str(r["name"]),"nameKey":str(r["name"]).lower(),"type1":str(r.type_1),"type2":str(r.type_2),
      "generation":int(r.generation),"legendary":bool(r.legendary),"origin":str(r.origen),"visualType":str(r.tipo_visual),
      "spriteLocal":"" if pd.isna(r.sprite_local) else str(r.sprite_local),"hp":float(r.hp),"attack":float(r.attack),"defense":float(r.defense),
      "sp_atk":float(r.sp_atk),"sp_def":float(r.sp_def),"speed":float(r.speed),"total":float(r.total),"avg":float(r.promedio_estadisticas)})
local={}
for r in records:
    if r["origin"]=="Artificial" and r["spriteLocal"]:
        b64=base64.b64encode((BASE/r["spriteLocal"]).read_bytes()).decode("ascii")
        local[r["nameKey"]]="data:image/png;base64,"+b64

template='<!DOCTYPE html>\n<html lang="es">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Scatter 3D Pokémon con sprites</title>\n<style>\n*{box-sizing:border-box} html,body{margin:0;height:100%;font-family:Inter,Segoe UI,Arial,sans-serif;background:#f7f9fc;color:#172033}\n#app{height:100vh;display:flex;flex-direction:column}\n#bar{background:#fff;border-bottom:1px solid #dbe1ea;padding:10px 14px;display:flex;gap:10px;align-items:center;flex-wrap:wrap;box-shadow:0 1px 7px rgba(15,23,42,.05)}\n#bar h1{font-size:17px;margin:0 auto 0 0;font-weight:700}\n.control{display:flex;align-items:center;gap:6px;font-size:12px;color:#475569}\nselect,button{padding:7px 9px;border:1px solid #cbd5e1;border-radius:8px;background:#fff;color:#172033} button{cursor:pointer}\n#status{font-size:12px;color:#64748b;min-width:210px;text-align:right}\n#wrap{position:relative;flex:1;min-height:460px;overflow:hidden}\ncanvas{width:100%;height:100%;display:block;cursor:grab} canvas:active{cursor:grabbing}\n#tip{display:none;position:absolute;z-index:5;pointer-events:none;background:rgba(255,255,255,.97);border:1px solid #cbd5e1;border-radius:10px;padding:9px 11px;box-shadow:0 8px 24px rgba(15,23,42,.18);font-size:12px;line-height:1.45;min-width:190px}\n#tip b{font-size:13px;color:#0f172a}.muted{color:#64748b}\n#legend{position:absolute;right:12px;top:12px;background:rgba(255,255,255,.92);border:1px solid #d8dee8;border-radius:9px;padding:8px 10px;font-size:11px;max-width:265px;box-shadow:0 3px 12px rgba(15,23,42,.08)}\n#help{position:absolute;left:12px;bottom:10px;background:rgba(255,255,255,.9);padding:7px 9px;border:1px solid #d8dee8;border-radius:8px;font-size:11px;color:#566176}\n</style>\n</head>\n<body>\n<div id="app">\n  <div id="bar">\n    <h1>Scatter 3D Pokémon · sprites oficiales + sprites locales</h1>\n    <div class="control"><label>Origen</label><select id="filterOrigin"><option value="all">Todos</option><option value="Artificial">Artificiales</option><option value="Oficial">Oficiales</option></select></div>\n    <div class="control"><label>Tipo</label><select id="filterType"><option value="all">Todos</option></select></div>\n    <div class="control"><label>Generación</label><select id="filterGen"><option value="all">Todas</option></select></div>\n    <div class="control"><label>Legendario</label><select id="filterLegendary"><option value="all">Todos</option><option value="yes">Sí</option><option value="no">No</option></select></div>\n    <div class="control"><label>Artificial</label><select id="filterArtificialName"><option value="all">Todos</option></select></div>\n    <div class="control"><label>Eje Z</label><select id="zMetric"><option value="avg">Promedio</option><option value="attack">Ataque</option><option value="sp_atk">Ataque especial</option><option value="defense">Defensa</option><option value="sp_def">Defensa especial</option><option value="speed">Velocidad</option><option value="hp">HP</option><option value="total">Total</option></select></div>\n    <div class="control"><label>Tamaño</label><select id="size"><option value="0.8">Pequeño</option><option value="1" selected>Normal</option><option value="1.25">Grande</option></select></div>\n    <button id="reset">Restablecer vista</button>\n    <span id="status"></span>\n  </div>\n  <div id="wrap">\n    <canvas id="c"></canvas><div id="tip"></div>\n    <div id="legend"><b>Sprites</b><br>Oficiales: GitHub · PokeAPI/sprites.<br>Artificiales: PNG de <b>sprites_normalizados/</b>, incrustados sin fondo y en lienzo 96×96.</div>\n    <div id="help">Arrastra para rotar · rueda para zoom · pasa el mouse sobre un sprite</div>\n  </div>\n</div>\n<script>\nconst DATA = __DATA__;\nconst LOCAL_SPRITES = __LOCAL__;\nconst GIT_BASE = \'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/\';\nconst canvas=document.getElementById(\'c\'), ctx=canvas.getContext(\'2d\'), wrap=document.getElementById(\'wrap\'), tip=document.getElementById(\'tip\');\nconst filterOrigin=document.getElementById(\'filterOrigin\'),filterType=document.getElementById(\'filterType\'),filterGen=document.getElementById(\'filterGen\'),filterLegendary=document.getElementById(\'filterLegendary\'),filterArtificialName=document.getElementById(\'filterArtificialName\'),zMetric=document.getElementById(\'zMetric\'),sizeSel=document.getElementById(\'size\'),statusEl=document.getElementById(\'status\');\nlet DPR=Math.max(1,Math.min(window.devicePixelRatio||1,2)),W=800,H=600,rotX=-0.62,rotY=.90,zoom=1,dragging=false,lastX=0,lastY=0;\nconst typeOrder=[\'Normal\',\'Fire\',\'Water\',\'Electric\',\'Grass\',\'Ice\',\'Fighting\',\'Poison\',\'Ground\',\'Flying\',\'Psychic\',\'Bug\',\'Rock\',\'Ghost\',\'Dragon\',\'Dark\',\'Steel\',\'Fairy\'].filter(t=>DATA.some(d=>d.type1===t));\nconst typePos=Object.fromEntries(typeOrder.map((t,i)=>[t,i]));\nconst imageCache=new Map(); let loaded=0,failed=0,totalUnique=0;\nfunction spriteSrc(p){ return p.origin===\'Artificial\' ? LOCAL_SPRITES[p.nameKey] : `${GIT_BASE}${p.dex}.png`; }\nconst points=DATA.map(d=>Object.assign({},d,{image:null,visible:true,screen:null,sizeMul:1}));\nfunction hashJitter(s){let h=2166136261;for(let i=0;i<s.length;i++){h^=s.charCodeAt(i);h=Math.imul(h,16777619)};return h>>>0}\nfunction modelPoint(d){\n const mid=(typeOrder.length-1)*42/2, metricMax=currentMaxMetric(); const h=hashJitter(`${d.dex}-${d.name}`);\n const jx=((h&255)/255-.5)*(d.origin===\'Artificial\'?34:20), jy=(((h>>>8)&255)/255-.5)*(d.origin===\'Artificial\'?24:14);\n return {x:(d.generation-1)*130-390+jx,y:(typePos[d.type1]??0)*42-mid+jy,z:(currentMetricValue(d)/metricMax)*320-160};\n}\nfunction project(pt){let{x,y,z}=pt;const cy=Math.cos(rotY),sy=Math.sin(rotY);let x1=x*cy-y*sy,y1=x*sy+y*cy,z1=z;const cx=Math.cos(rotX),sx=Math.sin(rotX);let y2=y1*cx-z1*sx,z2=y1*sx+z1*cx,x2=x1;const depth=780/(780+z2+220);return{x:W/2+x2*depth*zoom,y:H/2-y2*depth*zoom,scale:Math.max(.25,depth*zoom),depth:z2}}\nfunction currentMetricValue(d){const v=Number(d[zMetric.value]??d.avg??0);return isFinite(v)?v:0}\nfunction currentMetricLabel(){return({avg:\'Promedio\',attack:\'Ataque\',sp_atk:\'Ataque especial\',defense:\'Defensa\',sp_def:\'Defensa especial\',speed:\'Velocidad\',hp:\'HP\',total:\'Total\'})[zMetric.value]||zMetric.value}\nfunction currentMaxMetric(){let m=1;for(const d of DATA)m=Math.max(m,currentMetricValue(d));return m}\nfunction resize(){W=wrap.clientWidth;H=wrap.clientHeight;canvas.width=Math.floor(W*DPR);canvas.height=Math.floor(H*DPR);canvas.style.width=W+\'px\';canvas.style.height=H+\'px\';ctx.setTransform(DPR,0,0,DPR,0,0);ctx.imageSmoothingEnabled=false;draw()}\nfunction buildMenus(){\n [...new Set(DATA.map(d=>d.type1))].sort().forEach(t=>filterType.add(new Option(t,t)));\n [...new Set(DATA.map(d=>d.generation))].sort((a,b)=>a-b).forEach(g=>filterGen.add(new Option(String(g),String(g))));\n DATA.filter(d=>d.origin===\'Artificial\').map(d=>d.name).sort().forEach(n=>filterArtificialName.add(new Option(n,n)));\n}\nfunction passesFilters(p){if(filterOrigin.value!==\'all\'&&p.origin!==filterOrigin.value)return false;if(filterType.value!==\'all\'&&p.type1!==filterType.value&&p.type2!==filterType.value)return false;if(filterGen.value!==\'all\'&&String(p.generation)!==filterGen.value)return false;if(filterLegendary.value===\'yes\'&&!p.legendary)return false;if(filterLegendary.value===\'no\'&&p.legendary)return false;if(filterArtificialName.value!==\'all\'&&(p.origin!==\'Artificial\'||p.name!==filterArtificialName.value))return false;return true}\nfunction updateStatus(){const visible=points.filter(p=>p.visible).length;statusEl.textContent=`${visible} visibles · ${loaded}/${totalUnique} sprites cargados${failed?` · ${failed} fallaron`:\'\'}`}\nfunction applyControls(){const s=Number(sizeSel.value);for(const p of points){p.visible=passesFilters(p);p.sizeMul=s}updateStatus();draw()}\nfunction drawAxes(){\n const minY=-((typeOrder.length-1)*42/2)-25,maxY=((typeOrder.length-1)*42/2)+25;\n const axes=[[{x:-430,y:minY,z:-165},{x:430,y:minY,z:-165},\'#6c788e\'],[{x:-430,y:minY,z:-165},{x:-430,y:maxY,z:-165},\'#9aa6bd\'],[{x:-430,y:minY,z:-165},{x:-430,y:minY,z:190},\'#566176\']];\n ctx.lineWidth=1.15;for(const[a,b,c]of axes){const pa=project(a),pb=project(b);ctx.strokeStyle=c;ctx.beginPath();ctx.moveTo(pa.x,pa.y);ctx.lineTo(pb.x,pb.y);ctx.stroke()}\n ctx.fillStyle=\'#334155\';ctx.font=\'12px Arial\';for(let g=1;g<=7;g++){const p=project({x:(g-1)*130-390,y:minY-20,z:-165});ctx.fillText(String(g),p.x-3,p.y+14)}\n typeOrder.forEach((t,i)=>{const p=project({x:-455,y:i*42-((typeOrder.length-1)*42/2),z:-165});ctx.fillText(t,p.x-42,p.y+4)});ctx.fillText(currentMetricLabel(),14,18)\n}\nfunction draw(){ctx.clearRect(0,0,W,H);ctx.fillStyle=\'#f7f9fc\';ctx.fillRect(0,0,W,H);drawAxes();const drawable=points.filter(p=>p.visible).map(p=>{const s=project(modelPoint(p));p.screen=s;return[p,s]}).sort((a,b)=>a[1].depth-b[1].depth);\n for(const[p,s]of drawable){const im=p.image,scale=s.scale*p.sizeMul,size=Math.max(13,40*scale);if(im&&im.complete&&im.naturalWidth){ctx.drawImage(im,s.x-size/2,s.y-size/2,size,size)}}}\nfunction loadImage(src){if(imageCache.has(src))return imageCache.get(src);const promise=new Promise(resolve=>{const im=new Image();im.decoding=\'async\';im.onload=()=>{loaded++;updateStatus();resolve(im)};im.onerror=()=>{failed++;updateStatus();resolve(null)};im.src=src});imageCache.set(src,promise);return promise}\nasync function worker(queue){while(queue.length){const src=queue.pop();const im=await loadImage(src);for(const p of points)if(spriteSrc(p)===src)p.image=im;if((loaded+failed)%12===0)draw()}}\nasync function preloadAll(){const unique=[...new Set(points.map(spriteSrc).filter(Boolean))];totalUnique=unique.length;updateStatus();const q=unique.slice().reverse();await Promise.all(Array.from({length:Math.min(24,q.length)},()=>worker(q)));draw();updateStatus()}\nfunction hitTest(mx,my){const arr=points.filter(p=>p.visible&&p.screen&&p.image);for(let i=arr.length-1;i>=0;i--){const p=arr[i],s=p.screen,size=Math.max(13,40*s.scale*p.sizeMul);if(mx>=s.x-size/2&&mx<=s.x+size/2&&my>=s.y-size/2&&my<=s.y+size/2)return p}return null}\nfunction showTip(p,x,y){tip.innerHTML=`<b>${p.name}</b><br>${p.type1}${p.type2&&p.type2!==\'Sin segundo tipo\'?` / ${p.type2}`:\'\'}<br><span class="muted">Generación ${p.generation} · ${p.origin}</span><br>${currentMetricLabel()}: <b>${currentMetricValue(p).toFixed(2)}</b><br>HP ${p.hp} · Atq ${p.attack} · Def ${p.defense}<br>AtE ${p.sp_atk} · DfE ${p.sp_def} · Vel ${p.speed}`;tip.style.display=\'block\';tip.style.left=Math.min(W-220,x+14)+\'px\';tip.style.top=Math.max(8,y-20)+\'px\'}\ncanvas.addEventListener(\'mousedown\',e=>{dragging=true;lastX=e.clientX;lastY=e.clientY});window.addEventListener(\'mouseup\',()=>dragging=false);window.addEventListener(\'mousemove\',e=>{if(!dragging)return;rotY+=(e.clientX-lastX)*.008;rotX+=(e.clientY-lastY)*.008;lastX=e.clientX;lastY=e.clientY;draw()});\ncanvas.addEventListener(\'mousemove\',e=>{if(dragging){tip.style.display=\'none\';return}const r=canvas.getBoundingClientRect(),p=hitTest(e.clientX-r.left,e.clientY-r.top);p?showTip(p,e.clientX-r.left,e.clientY-r.top):tip.style.display=\'none\'});canvas.addEventListener(\'mouseleave\',()=>tip.style.display=\'none\');\ncanvas.addEventListener(\'wheel\',e=>{e.preventDefault();zoom*=e.deltaY<0?1.09:.92;zoom=Math.max(.45,Math.min(2.6,zoom));draw()},{passive:false});\n[filterOrigin,filterType,filterGen,filterLegendary,filterArtificialName,zMetric,sizeSel].forEach(el=>el.addEventListener(\'change\',applyControls));\ndocument.getElementById(\'reset\').onclick=()=>{rotX=-.62;rotY=.90;zoom=1;filterOrigin.value=filterType.value=filterGen.value=filterLegendary.value=filterArtificialName.value=\'all\';zMetric.value=\'avg\';sizeSel.value=\'1\';applyControls()};\nwindow.addEventListener(\'resize\',resize);buildMenus();resize();applyControls();preloadAll();\n</script>\n</body></html>'
html=template.replace("__DATA__",json.dumps(records,ensure_ascii=False,separators=(",",":"))).replace("__LOCAL__",json.dumps(local,separators=(",",":")))
(BASE/"graficascatter.html").write_text(html,encoding="utf-8")
print("Generado:",(BASE/"graficascatter.html").resolve())


Generado: C:\Users\derek\Desktop\ECBD_9A_IDGS_PRACTICAS_230892\Practica08\graficascatter.html


In [29]:
IFrame(src="practica07_scatter3d_pokemon.html", width="100%", height=780)


## 14. Exportación de la visualización en HTML

La visualización ya fue exportada como archivo HTML autónomo
(`practica07_scatter3d_pokemon.html`), incluyendo Plotly.js vía CDN. El archivo
conserva **todas** sus funciones interactivas (rotación 3D, zoom, hover con sprite,
filtros por generación/tipo/rango) al abrirse directamente en cualquier navegador,
sin depender de Jupyter.


In [30]:
ruta_html = Path("practica07_scatter3d_pokemon.html")
print("Archivo exportado:", ruta_html.resolve())
print("Tamaño:", round(ruta_html.stat().st_size / 1024, 1), "KB")
print("Existe:", ruta_html.exists())


Archivo exportado: C:\Users\derek\Desktop\ECBD_9A_IDGS_PRACTICAS_230892\Practica08\practica07_scatter3d_pokemon.html
Tamaño: 274.4 KB
Existe: True


## 15. Hallazgos e interpretación

**1. Los Dragón y legendarios dominan las estadísticas más altas.**
El tipo `Dragon` tiene, en promedio, las estadísticas más altas de todos los tipos
principales (~91.8 de promedio, frente a ~72.5 del promedio general), y los Pokémon
`legendary=True` promedian **105.1** frente a **69.5** de los no legendarios — una
brecha de más de 35 puntos que confirma el diseño intencional de los legendarios como
Pokémon superiores en estadísticas.

**2. Las formas Mega y Primal generan los valores atípicos superiores.**
Los cinco Pokémon con mayor promedio de estadísticas (`Mega Mewtwo X/Y`, `Mega
Rayquaza`, `Primal Kyogre`, `Primal Groudon`) son todas variantes *Mega/Primal*, no
Pokémon base. Esto se observa claramente como un grupo de puntos que se separa del
resto en la parte superior del gráfico 3D, indicando que estas mecánicas de juego
(introducidas en generaciones 3 y 6) están diseñadas para maximizar el poder de
combate temporalmente.

**3. La generación 4 tiene, en promedio, las estadísticas base más altas.**
Al agrupar por generación, la Generación 4 (`Diamond`/`Pearl`/`Platinum`) presenta el
promedio más alto (76.5), seguida de la 6 y la 3 (~72.7), mientras que la Generación 2
tiene el promedio más bajo (69.7) — un patrón conocido como *power creep*, donde
generaciones más recientes tienden a introducir Pokémon con estadísticas base
ligeramente mayores.

**4. Dos registros son atípicos por diseño, no por error de captura.**
Los registros `Sesni` (#722) y `Chuchin` (#723) presentan estadísticas fuera de todo
rango Pokémon real (ataque de 676 y defensa de 678, cuando el máximo real observado en
el dataset es 190 y 230 respectivamente) y generaciones inexistentes (67 y 32). Se
identificaron como registros personalizados/de práctica y se trataron por separado
para no distorsionar el análisis de generaciones y tipos oficiales.
